# 02 · Retrieval and the refusal gate

This notebook runs the retrieval half of the system on two questions of the evaluation set: one the corpus answers (A06) and one it does not (O06). It shows the passages, how dense retrieval and BM25 are fused, and the score the refusal gate decides on. It makes **no LLM call**, so it uses none of the daily OpenRouter quota.

Before running it: build the index with `python src/index.py` (about 3 hours on a laptop CPU, resumable). The cross-encoder is downloaded on first use.

In [1]:
import os
import sys

# The notebooks live in notebooks/; the project code is in src/.
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

In [2]:
import index
import rag
from bm25 import BM25
from rerank import GATE_RERANKER, gate_score, load_reranker

embedder = index.load_embedder()
collection = index.open_index()
bm25 = BM25(index.all_chunks(collection))
reranker = load_reranker(GATE_RERANKER)
print(collection.count(), "chunks in the index; gate cross-encoder:", GATE_RERANKER)

3410 chunks in the index; gate cross-encoder: minilm


In [3]:
import json

questions = {q["id"]: q for q in json.load(open(os.path.join(ROOT, "data", "eval", "questions.json"), encoding="utf-8"))}
answerable, out_of_corpus = questions["A06"], questions["O06"]
print("A06:", answerable["question"])
print("     expected:", answerable["ground_truth"], f"({answerable['source']}, p. {answerable['page']})")
print("O06:", out_of_corpus["question"])

A06: How long should parenteral artesunate be given to a patient with severe malaria?
     expected: Severe malaria is treated with intravenous or intramuscular artesunate for at least 24 hours and until the patient can tolerate oral medication. (WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 14)
O06: How long is a typical course of isotretinoin for severe acne?


## 1. Dense retrieval and BM25

Dense retrieval embeds the question with BGE-M3 and takes the closest chunks by cosine similarity. BM25 (written from scratch in `src/bm25.py`) ranks chunks by the question's words.

In [4]:
def show(hits, score_key="similarity"):
    """Print one line per hit: rank, score, citation label and the start of the text."""
    for i, h in enumerate(hits, 1):
        score = h.get(score_key)
        score = f"{score:.3f}" if score is not None else "  -  "
        print(f"[{i}] {score}  {rag.source_label(h)}  {h['text'][:70]!r}")


q = answerable["question"]
print("Dense, top 5:")
show(rag.retrieve(q, embedder, collection))
print("\nBM25, top 5:")
show(bm25.search(q, 5))

Dense, top 5:
[1] 0.804  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 79  '7.4.1 | ARTESUNATE Treating severe malaria Treat adults and children w'
[2] 0.796  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 75  'Treating severe malaria Treat adults and children with severe malaria '
[3] 0.762  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 177  'A4.12 Is parenteral artesunate superior to parenteral quinine in preve'
[4] 0.753  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 80  'Dosing of artesunate injection in severe malaria Revised dose recommen'
[5] 0.746  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 14  'tolerate oral therapy, complete treatment with 3 days of ACT (add sing'

BM25, top 5:
[1]   -    WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 90  'bo. 7.11 | TREATMENT OF SEVERE MALARIA DURING PREGNANCY Women in the s'
[2]   -    WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 78  'The degree of acidosis is an important det

## 2. Hybrid retrieval: reciprocal rank fusion

The 20 best of each list are fused by reciprocal rank fusion (Cormack et al., 2009): a chunk scores the sum of 1 / (60 + rank) over the lists it appears in. The first 5 are the passages.

In [5]:
dense = rag.retrieve(q, embedder, collection, k=rag.CANDIDATES)
hybrid = rag.rrf([dense, bm25.search(q, rag.CANDIDATES)])[:rag.TOP_K]
show(hybrid)
print("\nEvidence page", answerable["page"], "found:",
      any(h["source"] == answerable["source"] and h["page"] == answerable["page"] for h in hybrid))

[1] 0.804  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 79  '7.4.1 | ARTESUNATE Treating severe malaria Treat adults and children w'
[2] 0.762  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 177  'A4.12 Is parenteral artesunate superior to parenteral quinine in preve'
[3] 0.729  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 86  '7.7 | FOLLOW-ON TREATMENT The current recommendation of experts is to '
[4] 0.796  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 75  'Treating severe malaria Treat adults and children with severe malaria '
[5] 0.739  WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 80  'inetics of intravenous artesunate: a pooled analysis of individual dat'

Evidence page 14 found: False


Dense retrieval has page 14, the page the evaluation set names for A06, in fifth place; after fusion it drops out, and the passages come from pages 79, 177, 86, 75 and 80 of the same guideline. Page 79 states the same recommendation (artesunate for at least 24 h), so the answer is still supported, but the retrieval comparison counts A06 as a page miss.

## 3. The refusal gate

Retrieval always returns five passages, even for a question the corpus cannot answer, so the system needs a gate. The cross-encoder reads the question and each passage together and gives a score from 0 to 1; below 0.5 (the model's own boundary, not tuned on our questions) the system refuses without calling the LLM. The best cosine similarity is shown next to it: it was the first gate, at 0.6.

In [6]:
for item in (answerable, out_of_corpus):
    question = item["question"]
    dense = rag.retrieve(question, embedder, collection, k=rag.CANDIDATES)
    hits = rag.rrf([dense, bm25.search(question, rag.CANDIDATES)])[:rag.TOP_K]
    score = gate_score(question, hits, reranker)
    decision = "answer" if score >= rag.RERANK_THRESHOLD else "refuse"
    print(f"{item['id']}: best cosine {dense[0]['similarity']:.3f}   cross-encoder {score:.3f}   -> {decision}")

A06: best cosine 0.804   cross-encoder 0.999   -> answer
O06: best cosine 0.588   cross-encoder 0.156   -> refuse


O06 (isotretinoin for acne) comes close to the cosine threshold of 0.6 because it is a medical question, but the cross-encoder, which reads question and passage together, gives it a low score.

## 4. What the LLM would receive

For an answered question, the passages are numbered [1]–[5] in the prompt. A reply is kept only if it cites at least one of them and no other number; `[n]` is then replaced by the passage's source and page. The cell below builds the prompt and stops there.

In [7]:
prompt = rag.PROMPT.format(context=rag.format_context(hybrid), question=answerable["question"])
print(prompt[:1500], "...")

You are a medical assistant. Answer ONLY using the numbered passages below.
After every claim, cite the passage it comes from by its number in square
brackets, like [1] or [2][4]. Use only the passage numbers given.
If the passages do not contain the answer, say exactly:
"I cannot answer this question from the provided documents."
This is not medical advice. Do not diagnose. For doses, quote the passage and
tell the user to check with a clinician.

Passages:
[1] WHO_09_Guidelines_for_the_treatment_of_malaria.pdf, p. 79
7.4.1 | ARTESUNATE Treating severe malaria Treat adults and children with severe malaria (including infants, pregnant women in all trimesters and lactating women) with intravenous or intramuscular artesunate for at least 24 h. Once a patient has received at least 24 h of parenteral therapy and can tolerate oral therapy, complete treatment with 3 days of an ACT. Strong recommendation, high-quality evidence GRADE (see Annex 4, A4.12) In a systematic review of artesunate fo